In [56]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

In [57]:
df = pd.read_csv('/content/Dataset---Hate-Speech-Detection-using-Deep-Learning.csv')
df.head()

,class,tweet
0,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [58]:
df['class'].value_counts()

,count
class,
1,19190
2,4163
0,1430


In [59]:
print(df.isnull().sum())

class    0
tweet    0
dtype: int64


In [60]:
print(df.duplicated().sum())

0


In [61]:
df['text_length']= df['tweet'].apply(lambda x: len(x.split()))

print('Max Length:' , df['text_length'].max())

print('Min Length:' , df['text_length'].min())

print('Average Length:' , df['text_length'].mean())

df.sort_values(
    by='text_length',
    ascending = False
)[['tweet','text_length']].head(10)

Max Length: 52
Min Length: 1
Average Length: 14.117015696243392


,tweet,text_length
22478,Was finna slit my eyebrows up in the shop but ...,52
2099,"' She herd I was a dope boy , I herd she was a...",36
10377,I got no pic up lines I stay on my grind I tel...,33
11383,"I'm sippin on Patron nigga , my bitch bad to t...",33
19115,RT @hspiotta_21: c is for cunt\nu is for ur a ...,33
13699,"Oomf a hoe. She know she a hoe, I know she a h...",33
23174,Yo bitch a freak fucked ha to sleep and dat wa...,33
11158,I'd shut the fuck up or I'll just be wide and ...,33
20418,RT @yung_gleesh: If rather fukk a bitch dat fu...,32
24420,shout out to pullz cuz I kno if I was trash he...,32


In [62]:
def clean_text(text):
  text= text.lower()

  text = re.sub(r"http\S+\www\S+" , "", text)

  text = re.sub(r"<.&?>", "", text)

  text = re.sub(r"\s+", " ", text)

  return text.strip()

In [63]:
df['tweet'] = df['tweet'].apply(clean_text)

In [64]:
sentences = df['tweet'].values
labels = df['class'].values

X_train, X_test, y_train, y_test = train_test_split(
    sentences,
    labels,
    test_size = 0.2,
    random_state = 42,
    stratify = labels
)

In [65]:
vocab_size = 7000
oov_token = "<OOV>"

tokenizer = Tokenizer(
    num_words = vocab_size,
    oov_token = oov_token
)

tokenizer.fit_on_texts(X_train)

In [66]:
train_sequences = tokenizer.texts_to_sequences(X_train)

test_sequences = tokenizer.texts_to_sequences(X_test)

In [67]:
max_length = 15

X_train_pad = pad_sequences(
    train_sequences,
    maxlen = max_length,
    padding='post',
    truncating = 'post'
)

X_test_pad = pad_sequences(
    test_sequences,
    maxlen = max_length,
    padding='post',
    truncating = 'post'
)

In [68]:
from imblearn.over_sampling import RandomOverSampler\

ros = RandomOverSampler(sampling_strategy={0: 5000},random_state = 42)
X_train_pad, y_train = ros.fit_resample(X_train_pad, y_train)

In [69]:
from tensorflow.keras.layers import SimpleRNN
embedding_dim = 300

modelRNN = Sequential([

                     Input(shape=(max_length,)),

                     Embedding(
                         input_dim = vocab_size,
                         output_dim = embedding_dim
                     ),

                     SimpleRNN(64),

                     Dropout(0.3),

                     Dense(32, activation='relu'),

                     Dropout(0.3),

                     Dense(3, activation='softmax')
])

In [70]:
modelRNN.compile(
    optimizer = Adam(learning_rate=0.0005),
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

modelRNN.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 15, 300)        │     2,100,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 64)             │        23,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,125,539 (8.11 MB)

 Trainable params: 2,125,539 (8.11 MB)

 Non-trainable params: 0 (0.00 B)

In [71]:
historyRNN = modelRNN.fit(
    X_train_pad,
    y_train,
    epochs = 10,
    batch_size = 64,
    validation_data = (X_test_pad, y_test),
    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

Epoch 1/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.7575 - loss: 0.6155 - val_accuracy: 0.7962 - val_loss: 0.5002 - learning_rate: 5.0000e-04
Epoch 2/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9163 - loss: 0.2425 - val_accuracy: 0.8517 - val_loss: 0.4597 - learning_rate: 5.0000e-04
Epoch 3/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9595 - loss: 0.1290 - val_accuracy: 0.8455 - val_loss: 0.5108 - learning_rate: 5.0000e-04
Epoch 4/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9745 - loss: 0.0859 - val_accuracy: 0.8491 - val_loss: 0.6049 - learning_rate: 5.0000e-04
Epoch 5/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9826 - loss: 0.0582 - val_accuracy: 0.8487 - val_loss: 0.6764 - learning_rate: 5.0000e-04
Epoch 6/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9903 - loss: 0.0334 - val_accuracy: 0.8554 - val_loss: 0.7430 - learning_rate: 2.5000e-04
Epoch 7/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy

In [72]:
y_pred = modelRNN.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))
print(confusion_matrix(y_test, y_pred_classes))

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
              precision    recall  f1-score   support

           0       0.33      0.16      0.22       286
           1       0.92      0.91      0.91      3838
           2       0.67      0.84      0.74       833

    accuracy                           0.85      4957
   macro avg       0.64      0.64      0.62      4957
weighted avg       0.85      0.85      0.85      4957

[[  46  186   54]
 [  69 3477  292]
 [  25  109  699]]


In [73]:
model = Sequential([

                     Input(shape=(max_length,)),

                     Embedding(
                         input_dim = vocab_size,
                         output_dim = embedding_dim
                     ),

                     LSTM(64),

                     Dropout(0.3),

                     Dense(32, activation='relu'),

                     Dropout(0.3),

                     Dense(3, activation='softmax')
])

In [74]:
model.compile(
    optimizer = Adam(learning_rate = 0.0005),
    loss = 'sparse_categorical_crossentropy',
    metrics =["accuracy"]
)

model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 15, 300)        │     2,100,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 64)             │        93,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,195,619 (8.38 MB)

 Trainable params: 2,195,619 (8.38 MB)

 Non-trainable params: 0 (0.00 B)

In [75]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

historyLSTM =model.fit(
    X_train_pad,
    y_train,
    epochs = 10,
    batch_size = 64,
    validation_data = (X_test_pad, y_test),
    callbacks =[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

Epoch 1/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.7474 - loss: 0.6370 - val_accuracy: 0.8473 - val_loss: 0.3957 - learning_rate: 5.0000e-04
Epoch 2/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8839 - loss: 0.3241 - val_accuracy: 0.8501 - val_loss: 0.3988 - learning_rate: 5.0000e-04
Epoch 3/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9231 - loss: 0.2260 - val_accuracy: 0.8439 - val_loss: 0.4494 - learning_rate: 5.0000e-04
Epoch 4/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9438 - loss: 0.1718 - val_accuracy: 0.8390 - val_loss: 0.5086 - learning_rate: 5.0000e-04
Epoch 5/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9663 - loss: 0.1081 - val_accuracy: 0.8539 - val_loss: 0.6235 - learning_rate: 2.5000e-04
Epoch 6/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9746 - loss: 0.0854 - val_accuracy: 0.8533 - val_loss: 0.6628 - learning_rate: 2.5000e-04


In [76]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print('Loss:', loss)
print('Accuracy:', accuracy)

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8473 - loss: 0.3957
Loss: 0.3957296311855316
Accuracy: 0.8472866415977478


In [77]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test_pad)

y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))
print(confusion_matrix(y_test, y_pred_classes))

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
              precision    recall  f1-score   support

           0       0.32      0.49      0.39       286
           1       0.92      0.90      0.91      3838
           2       0.77      0.74      0.76       833

    accuracy                           0.85      4957
   macro avg       0.67      0.71      0.68      4957
weighted avg       0.86      0.85      0.85      4957

[[ 139  121   26]
 [ 238 3447  153]
 [  52  167  614]]


In [78]:
from tensorflow.keras.layers import Bidirectional

model2 = Sequential([

                     Input(shape=(max_length,)),

                     Embedding(
                         input_dim = vocab_size,
                         output_dim = embedding_dim
                     ),

                     Bidirectional(
                     LSTM(64)
                     ),

                     Dropout(0.3),

                     Dense(32, activation='relu'),

                     Dropout(0.3),

                     Dense(3, activation='softmax')
])

In [79]:
model2.compile(
    optimizer = Adam(learning_rate=0.0005),
    loss ='sparse_categorical_crossentropy',
    metrics =['accuracy']
)

model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 15, 300)        │     2,100,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 64)             │        93,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,586,859 (25.13 MB)

 Trainable params: 2,195,619 (8.38 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,391,240 (16.75 MB)

In [80]:
historyBi =model2.fit(
    X_train_pad,
    y_train,
    epochs = 10,
    batch_size = 64,
    validation_data = (X_test_pad, y_test),
    callbacks =[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

Epoch 1/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.7502 - loss: 0.6197 - val_accuracy: 0.8505 - val_loss: 0.3970 - learning_rate: 5.0000e-04
Epoch 2/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8859 - loss: 0.3123 - val_accuracy: 0.8550 - val_loss: 0.4044 - learning_rate: 5.0000e-04
Epoch 3/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9251 - loss: 0.2117 - val_accuracy: 0.8469 - val_loss: 0.4832 - learning_rate: 5.0000e-04
Epoch 4/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9487 - loss: 0.1511 - val_accuracy: 0.8475 - val_loss: 0.5610 - learning_rate: 5.0000e-04
Epoch 5/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9717 - loss: 0.0921 - val_accuracy: 0.8568 - val_loss: 0.7006 - learning_rate: 2.5000e-04
Epoch 6/10
371/371 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9792 - loss: 0.0710 - val_accuracy: 0.8471 - val_loss: 0.7805 - learning_rate: 2.5000e-04


In [81]:
loss, accuracy = model2.evaluate(X_test_pad, y_test)

print('Loss:', loss)
print('Accuracy:', accuracy)

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8505 - loss: 0.3970
Loss: 0.396965354681015
Accuracy: 0.8505144119262695


In [82]:
y_pred = model2.predict(X_test_pad)

y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))
print(confusion_matrix(y_test, y_pred_classes))

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
              precision    recall  f1-score   support

           0       0.35      0.49      0.41       286
           1       0.92      0.90      0.91      3838
           2       0.76      0.75      0.75       833

    accuracy                           0.85      4957
   macro avg       0.68      0.71      0.69      4957
weighted avg       0.86      0.85      0.86      4957

[[ 140  118   28]
 [ 215 3454  169]
 [  43  168  622]]


In [83]:
model2.save('bilstm_hate_speech.keras')

import pickle

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [84]:
from google.colab import files

files.download('bilstm_hate_speech.keras')
files.download('tokenizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>